In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")    
plt.rcParams['figure.figsize'] = (12, 6)



In [4]:
Data = {
    'Umur': [59, 68, 69, 53, 57, 47, 50, 53, 60, 77, 58, 70, 46, 74, 76, 74, 69, 76, 81, 57, 70, 50, 76, 59, 46, 62, 78, 71, 75, 69, 52, 61, 51, 49, 69, 84, 69, 51, 48, 66, 60, 50, 76, 78, 58, 82, 54, 83, 49, 79],
    'Berat Badan': [47, 76, 89, 57, 54, 82, 74, 82, 73, 63, 80, 67, 72, 53, 55, 83, 51, 79, 89, 65, 89, 67, 86, 76, 77, 84, 57, 64, 54, 67, 78, 63, 77, 87, 49, 77, 89, 79, 59, 67, 60, 45, 63, 66, 59, 62, 47, 72, 61, 69],
    'Tinggi Badan': [164, 171, 156, 170, 152, 155, 146, 157, 155, 164, 152, 145, 169, 166, 155, 181, 172, 159, 153, 171, 181, 173, 171, 171, 150, 182, 149, 163, 151, 147, 150, 151, 178, 178, 168, 154, 177, 176, 154, 183, 149, 157, 146, 153, 166, 169, 181, 152, 164, 167],
    'Pikun': [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0]
}

In [5]:
df = pd.DataFrame(Data)

In [6]:
df.drop(columns=['Pikun']).describe().T
print(f"Total Baris : {df.shape[0]}\nTotal Kolom : {df.shape[1]}")

print(f"""
Total Baris Duplikat : {df.duplicated().sum()}
Total Nilai Kosong (Nan) :  {df.isna().sum().sum()}
Total Nilai Unik :\n{df.nunique()}
""") 

Total Baris : 50
Total Kolom : 4

Total Baris Duplikat : 0
Total Nilai Kosong (Nan) :  0
Total Nilai Unik :
Umur            30
Berat Badan     32
Tinggi Badan    29
Pikun            2
dtype: int64



In [7]:
x = df.drop(columns=['Pikun'])
y = df['Pikun']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify=y
)

In [27]:
models = {
    "LogisticRegression" : LogisticRegression(),
    " Random Forest " : RandomForestClassifier(random_state=42),
    " Decision Tree" : DecisionTreeClassifier(random_state=42), 
}

In [28]:
result_list = []

for name, model in models.items():
    pipeline = Pipeline([
        ("Scaler", StandardScaler()),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:,1] if hasattr(pipeline, "predict_proba") else None

    acc = accuracy_score(y_test,y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else np.False_

    result_list.append({
            'Model': name,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1-Score': f1,
            'ROC-AUC': roc_auc
        })


df_results = pd.DataFrame(result_list).sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
print(df_results)   



                Model  Accuracy  Precision    Recall  F1-Score   ROC-AUC
0       Decision Tree       1.0        1.0  1.000000       1.0  1.000000
1  LogisticRegression       0.9        1.0  0.666667       0.8  0.952381
2      Random Forest        0.8        1.0  0.333333       0.5  0.952381
